# Кредитный скоринг (поиск гиперпараметров)

Подбор гиперпараметров всех пяти моделей через Optuna на одном отложенном фолде. Найденные значения вставляются в MODEL_PARAMS основного ноутбука. Структура моделей совпадает с боевой, поэтому параметры переносятся напрямую.

## Подготовка окружения

Подключение библиотек и настройка вычислений на видеокарте.

In [1]:
import os
import gc
import glob
import json
import time
import warnings

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as pads
import torch
import torch.nn as nn
from abc import ABC, abstractmethod
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')

DEVICE = 'cuda'
AMP_DTYPE = torch.bfloat16
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

print('torch', torch.__version__, '|', torch.cuda.get_device_name(0))

torch 2.0.1+cu118 | NVIDIA A100-SXM4-80GB


## Параметры

Основные настройки обучения и ансамбля.

In [2]:
SEED = 42
BATCH = 8192
np.random.seed(SEED)
torch.manual_seed(SEED)

NN_FOLDS = 5
GBDT_FOLDS = 5
N_SEEDS = 1

PATIENCE = 5
MODEL_EPOCHS = {
    'gru': 10,
    'transformer': 15,
    'cnn': 18,
    'transformer_hier': 15,
}

EMA_DECAY = 0.999
EMA_EVERY = 4
VAL_SUBSAMPLE = 100_000

MAX_LEN = 50
D_MODEL = 256
GRU_LAYERS = 2
TF_LAYERS = 3
TF_FF = 1024
MAX_LR = 2e-3

USE_CACHE = True
CACHE_VERSION = 'v2'
SAVE_MODELS = True
RESUME = True
FINETUNE = False

PL_EPOCHS = 5
PL_POS_Q = 0.8
PL_NEG_Q = 0.99
PL_MAX_NEG_MULT = 10

MODEL_PARAMS = {
    'xgboost': dict(learning_rate=0.02, max_depth=7, min_child_weight=5,
                    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, gamma=0.0),
    'gru': dict(d_model=256, gru_layers=2, dropout=0.2, max_lr=2e-3, weight_decay=1e-5),
    'transformer': dict(d_model=256, tf_layers=3, tf_ff=1024, dropout=0.1,
                        max_lr=2e-3, weight_decay=1e-5),
    'cnn': dict(d_model=256, tcn_levels=4, dropout=0.1, max_lr=2e-3, weight_decay=1e-5),
    'transformer_hier': dict(d_model=256, hier_layers=4, hier_hidden_mult=2, dropout=0.1,
                             max_lr=1.2e-3, weight_decay=0.02, grad_clip=1.0, label_smooth=0.02),
}

print('BATCH', BATCH, '| фолды', NN_FOLDS, '| сиды', N_SEEDS, '| resume', RESUME)
print('finetune', FINETUNE, '| эпохи по моделям:', MODEL_EPOCHS)

BATCH 8192 | фолды 5 | сиды 1 | resume True
finetune False | эпохи по моделям: {'gru': 10, 'transformer': 15, 'cnn': 18, 'transformer_hier': 15}


## Расположение файлов

Исходные данные находятся рядом с ноутбуком. Здесь же задаются папки для сохранения моделей и промежуточных результатов; если предобработка уже сохранена, повторная подготовка не требуется.

In [3]:
OUT = os.getcwd()
SAVE_DIR = os.path.join(OUT, 'models_tuning')
CACHE_DIR = os.path.join(OUT, 'cache')
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

CACHE_KEYS = [
    'X_agg', 'X_test', 'y', 'ids_tr', 'ids_te',
    'tr_first', 'tr_cnt', 'tr_feats', 'tr_rn',
    'te_first', 'te_cnt', 'te_feats', 'te_rn',
]

def cache_path(name):
    return os.path.join(CACHE_DIR, f'{CACHE_VERSION}_{name}')

cache_ready = (
    USE_CACHE
    and os.path.exists(cache_path('meta.json'))
    and all(os.path.exists(cache_path(key + '.npy')) for key in CACHE_KEYS)
)

TRAIN = os.path.join(OUT, 'train_data.parquet')
TEST = os.path.join(OUT, 'test_data.parquet')
TGT = os.path.join(OUT, 'train_target.csv')
SUB = os.path.join(OUT, 'sample_submission.csv')

print('кэш:', 'есть' if cache_ready else 'нет', '| рабочая папка:', OUT)

кэш: есть | рабочая папка: /home/jupyter/alpha


## Предобработка данных

Функции для загрузки данных, построения агрегированных признаков по истории клиента и формирования последовательностей продуктов фиксированной длины.

In [4]:
def load_parquet(path):
    dataset = pads.dataset(path, format='parquet')
    columns = {}
    for name in dataset.schema.names:
        array = dataset.to_table(columns=[name]).column(0).combine_chunks()
        dtype = pa.int32() if name == 'id' else pa.int8()
        columns[name] = array.cast(dtype).to_numpy(zero_copy_only=False)
        del array
        gc.collect()
    frame = pd.DataFrame(columns)
    del columns
    gc.collect()
    return frame

In [5]:
def aggregate(df):
    df = df.copy()

    paym_cols = [c for c in df.columns if c.startswith('enc_paym_')]
    if paym_cols:
        df['n_bad_paym'] = (df[paym_cols].values >= 3).sum(1).astype(np.int16)

    overdue_cols = [c for c in ['pre_loans5', 'pre_loans530', 'pre_loans3060',
                                'pre_loans6090', 'pre_loans90'] if c in df.columns]
    if overdue_cols:
        df['tot_overdue'] = df[overdue_cols].sum(1).astype(np.int16)

    if 'pre_loans_outstanding' in df and 'pre_loans_credit_limit' in df:
        limit = df['pre_loans_credit_limit'].astype(np.float32) + 1
        df['util_ratio'] = (df['pre_loans_outstanding'] / limit).astype(np.float32)
    if 'pre_loans_next_pay_summ' in df and 'pre_loans_credit_limit' in df:
        limit = df['pre_loans_credit_limit'].astype(np.float32) + 1
        df['nextpay_ratio'] = (df['pre_loans_next_pay_summ'] / limit).astype(np.float32)

    flag_cols = [c for c in df.columns if c.startswith('is_zero') or c.endswith('_flag')]
    feat_cols = [c for c in df.columns if c not in ('id', 'rn')]
    numeric_cols = [c for c in feat_cols if c not in flag_cols]

    grouped = df.groupby('id', sort=True)

    stats = grouped[numeric_cols].agg(['min', 'max', 'mean', 'std', 'sum'])
    stats.columns = [f'{col}_{fn}' for col, fn in stats.columns]
    flag_means = grouped[flag_cols].mean().add_suffix('_mean')
    counts = grouped.size().rename('n_records').to_frame()
    last_vals = df.loc[grouped['rn'].idxmax()].set_index('id')[feat_cols].add_suffix('_last')
    first_vals = df.loc[grouped['rn'].idxmin()].set_index('id')[feat_cols].add_suffix('_first')

    nunique_cols = [c for c in ['enc_loans_credit_type', 'enc_loans_credit_status']
                    if c in df.columns]
    parts = [counts, stats, flag_means, last_vals, first_vals]
    if nunique_cols:
        parts.append(grouped[nunique_cols].nunique().add_suffix('_nunique'))

    features = pd.concat(parts, axis=1).astype(np.float32).fillna(0.0)
    for col in numeric_cols:
        if f'{col}_last' in features.columns and f'{col}_first' in features.columns:
            features[f'{col}_delta'] = (
                features[f'{col}_last'].values - features[f'{col}_first'].values)

    del stats, flag_means, counts, last_vals, first_vals, grouped, df
    gc.collect()
    return features

In [6]:
def build_sequences(df, seq_cols):
    df = df.sort_values(['id', 'rn'], kind='stable')
    ids = df['id'].values
    feats = (df[seq_cols].values.astype(np.int16) + 1).astype(np.int8)
    rn = df['rn'].values.astype(np.int8)
    uniq, first_idx, counts = np.unique(ids, return_index=True, return_counts=True)
    return uniq, first_idx.astype(np.int64), counts.astype(np.int32), feats, rn


def build_padded(first, cnt, feats, rn, max_len):
    n_clients = len(first)
    n_rows, n_feat = feats.shape
    row_client = np.repeat(np.arange(n_clients), cnt)

    position = np.arange(n_rows) - first[row_client]
    keep = np.minimum(cnt, max_len)
    shifted = position - (cnt - keep)[row_client]
    visible = shifted >= 0

    padded_feats = np.zeros((n_clients, max_len, n_feat), dtype=np.int8)
    padded_rn = np.zeros((n_clients, max_len), dtype=np.int8)
    padded_feats[row_client[visible], shifted[visible]] = feats[visible]
    padded_rn[row_client[visible], shifted[visible]] = rn[visible]
    return padded_feats, padded_rn, keep.astype(np.int16)

## Подготовка данных

Данные либо загружаются из ранее сохранённого результата, либо собираются заново и сохраняются для следующих запусков. Подготовленные последовательности переносятся на видеокарту.

In [7]:
if cache_ready:
    X_agg_np = np.load(cache_path('X_agg.npy'))
    X_test_agg_np = np.load(cache_path('X_test.npy'))
    y = np.load(cache_path('y.npy'))
    ids_train = np.load(cache_path('ids_tr.npy'))
    ids_test = np.load(cache_path('ids_te.npy'))
    tr_first = np.load(cache_path('tr_first.npy'))
    tr_cnt = np.load(cache_path('tr_cnt.npy'))
    tr_feats = np.load(cache_path('tr_feats.npy'))
    tr_rn = np.load(cache_path('tr_rn.npy'))
    te_first = np.load(cache_path('te_first.npy'))
    te_cnt = np.load(cache_path('te_cnt.npy'))
    te_feats = np.load(cache_path('te_feats.npy'))
    te_rn = np.load(cache_path('te_rn.npy'))
    meta = json.load(open(cache_path('meta.json')))
    SEQ_COLS = meta['SEQ_COLS']
    VOCAB = {k: int(v) for k, v in meta['VOCAB'].items()}
    NF = meta['NF']
    feat_cols_agg = meta['feat']
    print('кэш загружен | X_agg', X_agg_np.shape)
else:
    target = pd.read_csv(TGT)
    train_df = load_parquet(TRAIN)

    SEQ_COLS = [c for c in train_df.columns if c not in ('id', 'rn')]
    VOCAB = {c: int(train_df[c].max()) + 2 for c in SEQ_COLS}
    NF = len(SEQ_COLS)

    train_agg = aggregate(train_df)
    ids_train = train_agg.index.values
    feat_cols_agg = train_agg.columns.tolist()
    X_agg_np = train_agg.values.astype(np.float32)
    del train_agg
    gc.collect()

    y = target.set_index('id').reindex(ids_train)['flag'].values.astype(np.int8)
    _, tr_first, tr_cnt, tr_feats, tr_rn = build_sequences(train_df, SEQ_COLS)
    del train_df
    gc.collect()

    test_df = load_parquet(TEST)
    test_agg = aggregate(test_df).reindex(columns=feat_cols_agg, fill_value=0.0)
    ids_test = test_agg.index.values
    X_test_agg_np = test_agg.values.astype(np.float32)
    del test_agg
    gc.collect()
    _, te_first, te_cnt, te_feats, te_rn = build_sequences(test_df, SEQ_COLS)
    del test_df
    gc.collect()

    saved = {
        'X_agg': X_agg_np, 'X_test': X_test_agg_np, 'y': y,
        'ids_tr': ids_train, 'ids_te': ids_test,
        'tr_first': tr_first, 'tr_cnt': tr_cnt, 'tr_feats': tr_feats, 'tr_rn': tr_rn,
        'te_first': te_first, 'te_cnt': te_cnt, 'te_feats': te_feats, 'te_rn': te_rn,
    }
    for name, value in saved.items():
        np.save(cache_path(name + '.npy'), value)
    json.dump({'SEQ_COLS': SEQ_COLS, 'VOCAB': VOCAB, 'NF': NF, 'feat': feat_cols_agg},
              open(cache_path('meta.json'), 'w'))
    print('данные построены и сохранены в', CACHE_DIR)

n_test = len(te_first)

кэш загружен | X_agg (2100000, 457)


In [8]:
def to_gpu(padded):
    return torch.from_numpy(padded).to(DEVICE)

Ftr, Rtr, Ltr = (to_gpu(arr) for arr in build_padded(tr_first, tr_cnt, tr_feats, tr_rn, MAX_LEN))
Fte, Rte, Lte = (to_gpu(arr) for arr in build_padded(te_first, te_cnt, te_feats, te_rn, MAX_LEN))
y_gpu = torch.as_tensor(y, dtype=torch.float32, device=DEVICE)

TR_F, TR_R, TR_L, TR_Yg, TR_Y = Ftr, Rtr, Ltr, y_gpu, y

del tr_feats, tr_rn, te_feats, te_rn
gc.collect()
print('на GPU: train', tuple(Ftr.shape), '| test', tuple(Fte.shape),
      '| агрегатов', X_agg_np.shape[1], '| n_test', n_test)

на GPU: train (2100000, 50, 59) | test (900000, 50, 59) | агрегатов 457 | n_test 900000


## Базовый класс модели

Общая логика обучения с кросс-валидацией: обучение по частям с сохранением и переиспользованием уже посчитанного. Здесь также усреднение весов сети для более устойчивого результата.

In [9]:
class BaseModel(ABC):
    name = 'base'
    DEFAULTS = {}

    def __init__(self, params=None):
        self.params = dict(self.DEFAULTS)
        if params:
            self.params.update({k: v for k, v in params.items() if v is not None})

    @abstractmethod
    def fit_fold(self, train_idx, val_idx):
        ...

    @abstractmethod
    def predict(self, idx, which):
        ...

    def save(self, path):
        pass

    def load(self, path):
        pass

    def _ckpt_exists(self, path):
        return False

    def finetune(self, train_idx, val_idx):
        self.fit_fold(train_idx, val_idx)

    def cross_validate(self, y, folds, n_test, save_dir=None, resume=False, do_finetune=False):
        self._K = len(folds)
        self._k = 0
        self._ep_times = []
        self._ep_count = 0

        oof = np.zeros(len(y))
        test = np.zeros(n_test)
        fold_aucs = []
        start = time.time()

        for k, (train_idx, val_idx) in enumerate(folds):
            self._k = k + 1
            t0 = time.time()
            path = os.path.join(save_dir, f'{self.name}_fold{k}') if save_dir else None

            if resume and path and self._ckpt_exists(path):
                self.load(path)
                if do_finetune:
                    print(f'  [{self.name}] фолд {k + 1}/{len(folds)}: дообучаю')
                    self.finetune(train_idx, val_idx)
                    if path:
                        self.save(path)
                else:
                    print(f'  [{self.name}] фолд {k + 1}/{len(folds)}: загружен из models/')
            else:
                print(f'  [{self.name}] фолд {k + 1}/{len(folds)}: обучаю с нуля')
                self.fit_fold(train_idx, val_idx)
                if path:
                    self.save(path)

            oof[val_idx] = self.predict(val_idx, 'train')
            test += self.predict(None, 'test') / len(folds)

            fold_auc = roc_auc_score(y[val_idx], oof[val_idx])
            fold_aucs.append(fold_auc)
            print(f'  [{self.name}] фолд {k + 1}/{len(folds)}  '
                  f'AUC={fold_auc:.4f}  ({time.time() - t0:.0f}s)')

            gc.collect()
            torch.cuda.empty_cache()

        auc = roc_auc_score(y, oof)
        print(f'== {self.name}: OOF AUC = {auc:.4f}  '
              f'(фолды {np.mean(fold_aucs):.4f} +/- {np.std(fold_aucs):.4f}) | '
              f'{(time.time() - start) / 60:.1f} мин')
        return {'name': self.name, 'oof': oof, 'test': test, 'auc': float(auc)}

## Нейросетевые архитектуры

Несколько разных сетей, работающих с последовательностью продуктов клиента: рекуррентная, на основе аттеншн и свёрточная. Отдельная архитектура дополнительно учитывает помесячную историю платежей внутри каждого продукта.

In [10]:
def pick_nhead(d_model, candidates=(8, 6, 4, 2, 1)):
    return max(h for h in candidates if d_model % h == 0)


def masked_mean_max(out, mask):
    weights = mask.unsqueeze(-1).float()
    mean = (out * weights).sum(1) / weights.sum(1).clamp(min=1)
    maximum = out.masked_fill(mask.unsqueeze(-1) == 0, -1e4).max(1).values
    return mean, maximum


class FeatEmbed(nn.Module):
    def __init__(self, vocab_list, emb_dim=16, rn_vocab=64, d_model=256):
        super().__init__()
        offsets = np.cumsum([0] + list(vocab_list[:-1])).astype('int64')
        self.register_buffer('offsets', torch.tensor(offsets))
        self.nf = len(vocab_list)
        self.table = nn.Embedding(int(sum(vocab_list)), emb_dim)
        self.rn_emb = nn.Embedding(rn_vocab, emb_dim)
        self.proj = nn.Sequential(
            nn.Linear(emb_dim * (self.nf + 1), d_model),
            nn.LayerNorm(d_model),
            nn.ReLU(),
            nn.Dropout(0.1),
        )

    def forward(self, features, rn):
        x = self.table(features + self.offsets).flatten(2)
        x = torch.cat([x, self.rn_emb(rn)], dim=-1)
        return self.proj(x)

In [11]:
class GRUNet(nn.Module):
    def __init__(self, vocab_list, d_model=256, hidden=256, layers=2, dropout=0.2):
        super().__init__()
        self.emb = FeatEmbed(vocab_list, d_model=d_model)
        self.gru = nn.GRU(
            d_model, hidden, num_layers=layers, batch_first=True,
            bidirectional=True, dropout=dropout if layers > 1 else 0.0,
        )
        out_dim = hidden * 2
        self.head = nn.Sequential(
            nn.Linear(out_dim * 3, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
        )

    def forward(self, features, rn, mask):
        x = self.emb(features, rn)
        lengths = mask.sum(1).clamp(min=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths, batch_first=True, enforce_sorted=False)
        out, _ = self.gru(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)
        mean, maximum = masked_mean_max(out, mask[:, :out.size(1)])
        last_idx = (lengths - 1).to(out.device).long()
        last = out[torch.arange(out.size(0), device=out.device), last_idx]
        return self.head(torch.cat([mean, maximum, last], dim=-1))


class TransformerNet(nn.Module):
    def __init__(self, vocab_list, d_model=256, nhead=8, nlayers=3, dim_ff=1024, dropout=0.1):
        super().__init__()
        self.emb = FeatEmbed(vocab_list, d_model=d_model)
        layer = nn.TransformerEncoderLayer(
            d_model, nhead, dim_feedforward=dim_ff, dropout=dropout,
            batch_first=True, activation='gelu',
        )
        self.enc = nn.TransformerEncoder(layer, nlayers, enable_nested_tensor=False)
        self.head = nn.Sequential(
            nn.Linear(d_model * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
        )

    def forward(self, features, rn, mask):
        x = self.emb(features, rn)
        out = self.enc(x, src_key_padding_mask=(mask == 0))
        mean, maximum = masked_mean_max(out, mask)
        return self.head(torch.cat([mean, maximum], dim=-1))

In [12]:
class TCNBlock(nn.Module):
    def __init__(self, channels, kernel=3, dilation=1, dropout=0.1):
        super().__init__()
        pad = (kernel - 1) * dilation // 2
        self.conv1 = nn.Conv1d(channels, channels, kernel, padding=pad, dilation=dilation)
        self.conv2 = nn.Conv1d(channels, channels, kernel, padding=pad, dilation=dilation)
        self.bn1 = nn.BatchNorm1d(channels)
        self.bn2 = nn.BatchNorm1d(channels)
        self.act = nn.GELU()
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        y = self.drop(self.act(self.bn1(self.conv1(x))))
        y = self.drop(self.act(self.bn2(self.conv2(y))))
        return self.act(x + y)


class TCNNet(nn.Module):
    def __init__(self, vocab_list, d_model=256, channels=256, dilations=(1, 2, 4, 8), dropout=0.1):
        super().__init__()
        self.emb = FeatEmbed(vocab_list, d_model=d_model)
        self.inp = nn.Conv1d(d_model, channels, 1)
        self.blocks = nn.ModuleList([TCNBlock(channels, 3, d, dropout) for d in dilations])
        self.head = nn.Sequential(
            nn.Linear(channels * 2, 256),
            nn.GELU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
        )

    def forward(self, features, rn, mask):
        x = self.emb(features, rn) * mask.unsqueeze(-1).float()
        x = self.inp(x.transpose(1, 2))
        for block in self.blocks:
            x = block(x)
        x = x.transpose(1, 2)
        mean, maximum = masked_mean_max(x, mask)
        return self.head(torch.cat([mean, maximum], dim=-1))

In [13]:
class AttnPool(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.q = nn.Parameter(torch.zeros(1, 1, d))
        nn.init.normal_(self.q, std=0.02)
        self.scale = d ** -0.5

    def forward(self, x, mask):
        scores = (x @ self.q.transpose(1, 2)).squeeze(-1) * self.scale
        weights = scores.masked_fill(mask == 0, -1e4).softmax(1).unsqueeze(-1)
        return (x * weights).sum(1)


class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.w = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        scale = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x * scale * self.w


class GEGLU(nn.Module):
    def __init__(self, d, hidden):
        super().__init__()
        self.w12 = nn.Linear(d, hidden * 2)
        self.w3 = nn.Linear(hidden, d)

    def forward(self, x):
        a, b = self.w12(x).chunk(2, dim=-1)
        return self.w3(torch.nn.functional.gelu(a) * b)


class LlamaBlock(nn.Module):
    def __init__(self, d, nhead, hidden, dropout=0.1):
        super().__init__()
        self.n1 = RMSNorm(d)
        self.attn = nn.MultiheadAttention(d, nhead, dropout=dropout, batch_first=True)
        self.n2 = RMSNorm(d)
        self.ff = GEGLU(d, hidden)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask):
        h = self.n1(x)
        attended, _ = self.attn(h, h, h, key_padding_mask=key_padding_mask, need_weights=False)
        x = x + self.drop(attended)
        x = x + self.drop(self.ff(self.n2(x)))
        return x


class MonthEncoder(nn.Module):
    def __init__(self, vocab, n_months, emb=12, d_out=48, nhead=4):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb)
        self.pos = nn.Embedding(n_months, emb)
        layer = nn.TransformerEncoderLayer(
            emb, nhead, emb * 2, 0.1, batch_first=True, activation='gelu', norm_first=True,
        )
        self.enc = nn.TransformerEncoder(layer, 1, enable_nested_tensor=False)
        self.proj = nn.Linear(emb * 2, d_out)

    def forward(self, months):
        batch, n_products, n_months = months.shape
        embedded = self.emb(months.reshape(batch * n_products, n_months))
        embedded = embedded + self.pos(torch.arange(n_months, device=months.device))[None]
        encoded = self.enc(embedded)
        pooled = torch.cat([encoded.mean(1), encoded.max(1).values], -1)
        return self.proj(pooled).reshape(batch, n_products, -1)

In [14]:
class TransformerHierNet(nn.Module):
    def __init__(self, vocab_list, seq_cols, d_model=256, nhead=8, nlayers=4,
                 hidden=512, dropout=0.1, max_len=50):
        super().__init__()
        self.paym_idx = [i for i, c in enumerate(seq_cols) if c.startswith('enc_paym_')]
        self.other_idx = [i for i in range(len(seq_cols)) if i not in self.paym_idx]

        other_vocab = [vocab_list[i] for i in self.other_idx]
        offsets = np.cumsum([0] + list(other_vocab[:-1])).astype('int64')
        self.register_buffer('off', torch.tensor(offsets))
        self.emb_other = nn.Embedding(int(sum(other_vocab)), 16)

        paym_vocab = max(vocab_list[i] for i in self.paym_idx) if self.paym_idx else 2
        self.month = MonthEncoder(int(paym_vocab), max(1, len(self.paym_idx)), emb=12, d_out=48)
        self.rn = nn.Embedding(64, 16)

        in_dim = 16 * len(self.other_idx) + 48 + 16
        self.proj = nn.Sequential(
            nn.Linear(in_dim, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.normal_(self.cls, std=0.02)
        self.blocks = nn.ModuleList(
            [LlamaBlock(d_model, nhead, hidden, dropout) for _ in range(nlayers)])
        self.norm = RMSNorm(d_model)
        self.attn = AttnPool(d_model)
        self.head = nn.Sequential(
            nn.Linear(d_model * 4, 256),
            nn.GELU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
        )

    def forward(self, features, rn, mask):
        batch = features.size(0)
        other = self.emb_other(features[:, :, self.other_idx] + self.off).flatten(2)
        months = self.month(features[:, :, self.paym_idx])
        tokens = self.proj(torch.cat([other, months, self.rn(rn)], dim=-1))

        tokens = torch.cat([self.cls.expand(batch, -1, -1), tokens], 1)
        full_mask = torch.cat([torch.ones(batch, 1, dtype=torch.bool, device=mask.device), mask], 1)
        for block in self.blocks:
            tokens = block(tokens, full_mask == 0)

        tokens = self.norm(tokens)
        cls_out = tokens[:, 0]
        seq = tokens[:, 1:]
        mean, maximum = masked_mean_max(seq, mask)
        return self.head(torch.cat([cls_out, self.attn(seq, mask), mean, maximum], dim=-1))

## Модели

Модели с одним интерфейсом обучения и предсказания. Для нейросетей предусмотрены ранняя остановка и аккуратное дообучение.

In [15]:
from xgboost import XGBClassifier


class XGBModel(BaseModel):
    name = 'xgboost'
    DEFAULTS = dict(
        learning_rate=0.02, max_depth=7, min_child_weight=5,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, gamma=0.0,
    )

    def _params(self):
        params = dict(self.params)
        pos = max(float((y == 1).sum()), 1.0)
        params.setdefault('scale_pos_weight', float((y == 0).sum()) / pos)
        params.update(
            n_estimators=6000, max_bin=256, eval_metric='auc',
            early_stopping_rounds=200, random_state=SEED, n_jobs=-1,
            tree_method='gpu_hist',
        )
        return params

    def fit_fold(self, train_idx, val_idx):
        self.model = XGBClassifier(**self._params())
        self.model.fit(
            X_agg_np[train_idx], y[train_idx],
            eval_set=[(X_agg_np[val_idx], y[val_idx])], verbose=False,
        )

    def _ckpt_exists(self, path):
        return os.path.exists(path + '.json')

    def predict(self, idx, which):
        matrix = X_test_agg_np if which == 'test' else X_agg_np[idx]
        return self.model.predict_proba(matrix)[:, 1]

    def save(self, path):
        self.model.save_model(path + '.json')

    def load(self, path):
        self.model = XGBClassifier()
        self.model.load_model(path + '.json')

    def finetune(self, train_idx, val_idx):
        booster = self.model.get_booster()
        self.model = XGBClassifier(**self._params())
        self.model.fit(
            X_agg_np[train_idx], y[train_idx],
            eval_set=[(X_agg_np[val_idx], y[val_idx])], verbose=False, xgb_model=booster,
        )

In [16]:
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.step = 0
        self.shadow = {k: v.detach().clone().float() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        self.step += 1
        decay = min(self.decay, (1.0 + self.step) / (10.0 + self.step))
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(decay).add_(v.detach().float(), alpha=1 - decay)
            else:
                self.shadow[k].copy_(v)

    @torch.no_grad()
    def copy_to(self, model):
        for k, v in model.state_dict().items():
            v.copy_(self.shadow[k].to(v.dtype))


class SeqModel(BaseModel):
    DEFAULTS = dict(d_model=None, gru_layers=None, tf_layers=None, tf_ff=None,
                    max_lr=None, weight_decay=1e-5)

    def __init__(self, params=None):
        super().__init__(params)
        self.epochs = MODEL_EPOCHS.get(self.name, 20)
        self.n_seeds = N_SEEDS
        self.init_state = None
        self.bs = None
        self._K = 1
        self._k = 1
        self._ep_times = []
        self._ep_count = 0

    def build_net(self):
        raise NotImplementedError

    def _slice(self, features, rn, lengths, idx):
        f = features[idx].long()
        r = rn[idx].long()
        length = lengths[idx].long()
        mask = torch.arange(f.shape[1], device=DEVICE)[None, :] < length[:, None]
        return f, r, mask

    @torch.no_grad()
    def _infer_net(self, net, features, rn, lengths, idx):
        net.eval()
        batch_size = self.bs or BATCH
        chunks = []
        for i in range(0, len(idx), batch_size):
            f, r, mask = self._slice(features, rn, lengths, idx[i:i + batch_size])
            with torch.autocast('cuda', dtype=AMP_DTYPE):
                prob = torch.sigmoid(net(f, r, mask).squeeze(1))
            chunks.append(prob.float().cpu().numpy())
        return np.concatenate(chunks)

    def _infer(self, features, rn, lengths, idx):
        per_net = [self._infer_net(net, features, rn, lengths, idx) for net in self.nets]
        return np.mean(per_net, axis=0)

    def _make_scheduler(self, optimizer, steps_per_epoch, lr_max):
        if self.init_state is not None:
            return torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=steps_per_epoch * self.epochs, eta_min=lr_max * 0.02)
        return torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=lr_max, steps_per_epoch=steps_per_epoch, epochs=self.epochs)

    def _run_epoch(self, net, optimizer, scheduler, ema, loss_fn,
                   train_idx, label_smooth, grad_clip):
        net.train()
        order = train_idx[torch.randperm(len(train_idx), device=DEVICE)]
        for i in range(0, len(order), self.bs):
            batch = order[i:i + self.bs]
            f, r, mask = self._slice(TR_F, TR_R, TR_L, batch)
            labels = TR_Yg[batch]
            if label_smooth > 0:
                labels = labels * (1 - label_smooth) + 0.5 * label_smooth

            optimizer.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=AMP_DTYPE):
                loss = loss_fn(net(f, r, mask).squeeze(1), labels)
            loss.backward()
            if grad_clip:
                torch.nn.utils.clip_grad_norm_(net.parameters(), grad_clip)
            optimizer.step()
            scheduler.step()

            self._gstep += 1
            if self._gstep % EMA_EVERY == 0:
                ema.update(net)

    def _evaluate(self, net, ema, ema_net, val_idx, val_labels):
        auc_raw = roc_auc_score(val_labels, self._infer_net(net, TR_F, TR_R, TR_L, val_idx))
        ema.copy_to(ema_net)
        auc_ema = roc_auc_score(val_labels, self._infer_net(ema_net, TR_F, TR_R, TR_L, val_idx))
        if auc_ema >= auc_raw:
            return auc_ema, ema_net, auc_raw, auc_ema
        return auc_raw, net, auc_raw, auc_ema

    def _train_one(self, train_idx, val_idx, seed):
        torch.manual_seed(SEED + seed)
        net = self.build_net().to(DEVICE)
        if self.init_state is not None:
            net.load_state_dict(self.init_state)
        ema = EMA(net, EMA_DECAY)
        ema_net = self.build_net().to(DEVICE)

        train_t = torch.as_tensor(train_idx, device=DEVICE)
        val_eval = val_idx if len(val_idx) <= VAL_SUBSAMPLE else val_idx[:VAL_SUBSAMPLE]
        val_eval_t = torch.as_tensor(val_eval, device=DEVICE)
        val_labels = TR_Y[val_eval]

        pos = max(float(TR_Y[train_idx].sum()), 1.0)
        neg = float((TR_Y[train_idx] == 0).sum())
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(neg / pos, device=DEVICE))

        lr_max = self.params.get('max_lr') or MAX_LR
        weight_decay = self.params.get('weight_decay') or 1e-5
        optimizer = torch.optim.AdamW(net.parameters(), lr=lr_max, weight_decay=weight_decay)
        grad_clip = self.params.get('grad_clip')
        label_smooth = self.params.get('label_smooth') or 0.0
        steps_per_epoch = max(1, (len(train_t) + self.bs - 1) // self.bs)
        scheduler = self._make_scheduler(optimizer, steps_per_epoch, lr_max)

        best_auc = 0.0
        best_state = None
        patience = 0
        self._gstep = 0

        for epoch in range(self.epochs):
            epoch_start = time.time()
            self._run_epoch(net, optimizer, scheduler, ema, loss_fn,
                            train_t, label_smooth, grad_clip)
            auc, source, auc_raw, auc_ema = self._evaluate(
                net, ema, ema_net, val_eval_t, val_labels)
            self._log_epoch(seed, epoch, auc, auc_raw, auc_ema, time.time() - epoch_start)

            if auc > best_auc:
                best_auc = auc
                best_state = {k: v.detach().cpu().clone() for k, v in source.state_dict().items()}
                patience = 0
            else:
                patience += 1
                if patience >= PATIENCE:
                    break

        final = self.build_net().to(DEVICE)
        final.load_state_dict(best_state)
        return final, best_auc

    def _log_epoch(self, seed, epoch, auc, auc_raw, auc_ema, seconds):
        self._ep_times.append(seconds)
        self._ep_count += 1
        avg = sum(self._ep_times) / len(self._ep_times)
        planned = self._K * self.n_seeds * self.epochs
        eta = max(0, planned - self._ep_count) * avg
        print(f'  {self.name} фолд {self._k}/{self._K} сид {seed + 1}/{self.n_seeds} '
              f'эпоха {epoch + 1}/{self.epochs} | val={auc:.4f} '
              f'(raw {auc_raw:.4f} / ema {auc_ema:.4f}) | {seconds:.0f}s | ~{eta / 60:.1f} мин')

    def fit_fold(self, train_idx, val_idx):
        self.nets = []
        self.bs = BATCH
        for seed in range(self.n_seeds):
            net, _ = self._train_one(train_idx, val_idx, seed)
            self.nets.append(net)

    def finetune(self, train_idx, val_idx):
        loaded = self.nets
        self.nets = []
        self.bs = BATCH
        base_lr = self.params.get('max_lr')
        self.params['max_lr'] = (base_lr or MAX_LR) / 5.0
        for seed, net in enumerate(loaded):
            self.init_state = net.state_dict()
            tuned, _ = self._train_one(train_idx, val_idx, seed)
            self.nets.append(tuned)
        self.init_state = None
        self.params['max_lr'] = base_lr

    def _ckpt_exists(self, path):
        return len(glob.glob(f'{path}_seed*.pt')) > 0

    def predict(self, idx, which):
        if which == 'test':
            test_idx = torch.arange(Fte.shape[0], device=DEVICE)
            return self._infer(Fte, Rte, Lte, test_idx)
        return self._infer(TR_F, TR_R, TR_L, torch.as_tensor(idx, device=DEVICE))

    def save(self, path):
        for i, net in enumerate(self.nets):
            torch.save(net.state_dict(), f'{path}_seed{i}.pt')

    def load(self, path):
        self.nets = []
        self.bs = BATCH
        for file in sorted(glob.glob(f'{path}_seed*.pt'))[:self.n_seeds]:
            net = self.build_net().to(DEVICE)
            net.load_state_dict(torch.load(file, map_location=DEVICE))
            self.nets.append(net)

In [17]:
class GRUModel(SeqModel):
    name = 'gru'

    def build_net(self):
        d = self.params.get('d_model') or D_MODEL
        layers = self.params.get('gru_layers') or GRU_LAYERS
        dropout = self.params.get('dropout', 0.2)
        return GRUNet([VOCAB[c] for c in SEQ_COLS], d, d, layers, dropout)


class TransformerModel(SeqModel):
    name = 'transformer'

    def build_net(self):
        d = self.params.get('d_model') or D_MODEL
        layers = self.params.get('tf_layers') or TF_LAYERS
        ff = self.params.get('tf_ff') or TF_FF
        dropout = self.params.get('dropout', 0.1)
        return TransformerNet([VOCAB[c] for c in SEQ_COLS], d, pick_nhead(d), layers, ff, dropout)


class TCNModel(SeqModel):
    name = 'cnn'

    def build_net(self):
        d = self.params.get('d_model') or D_MODEL
        levels = self.params.get('tcn_levels') or 4
        dropout = self.params.get('dropout', 0.1)
        dilations = tuple(2 ** i for i in range(levels))
        return TCNNet([VOCAB[c] for c in SEQ_COLS], d, d, dilations, dropout)

In [18]:
class TransformerHierModel(SeqModel):
    name = 'transformer_hier'

    def __init__(self, params=None):
        defaults = dict(d_model=256, max_lr=1.2e-3, weight_decay=0.02,
                        grad_clip=1.0, label_smooth=0.02)
        if params:
            defaults.update(params)
        super().__init__(defaults)

    def build_net(self):
        d = self.params.get('d_model') or 256
        nhead = pick_nhead(d, (8, 4, 2, 1))
        nlayers = self.params.get('hier_layers') or 4
        hidden = d * (self.params.get('hier_hidden_mult') or 2)
        dropout = self.params.get('dropout', 0.1)
        vocab = [VOCAB[c] for c in SEQ_COLS]
        return TransformerHierNet(vocab, SEQ_COLS, d, nhead, nlayers, hidden, dropout, MAX_LEN)

## Обучение по фолдам

Вспомогательная процедура, которая обучает модель по фолдам и сохраняет её предсказания для дальнейшего объединения.

In [19]:
def make_folds(n_splits):
    splitter = StratifiedKFold(n_splits, shuffle=True, random_state=SEED)
    return list(splitter.split(np.zeros(len(y)), y))

folds_gbdt = make_folds(GBDT_FOLDS)
folds_nn = make_folds(NN_FOLDS)

SAVE_TO = SAVE_DIR if SAVE_MODELS else None
PRED_DIR = os.path.join(SAVE_DIR, 'preds')
os.makedirs(PRED_DIR, exist_ok=True)

MODELS = ['xgboost', 'gru', 'transformer', 'cnn', 'transformer_hier']
runs = globals().get('runs', {})


def save_preds(name, result):
    np.savez(os.path.join(PRED_DIR, f'{name}.npz'),
             oof=result['oof'], test=result['test'], auc=result['auc'])


def load_saved_preds(names):
    for name in names:
        path = os.path.join(PRED_DIR, f'{name}.npz')
        if name not in runs and os.path.exists(path):
            data = np.load(path)
            runs[name] = {'oof': data['oof'], 'test': data['test'], 'auc': float(data['auc'])}


def run_model(name, model_class, folds):
    result = model_class(MODEL_PARAMS.get(name)).cross_validate(
        y, folds, n_test, SAVE_TO, RESUME, FINETUNE)
    runs[name] = result
    save_preds(name, result)
    print(f'>> {name}: OOF AUC={result["auc"]:.5f} | сохранено в preds/{name}.npz')
    return result

## Поиск гиперпараметров

Для каждой модели короткий поиск на одном фолде: сети обучаются ограниченное число эпох, а бустинг с ранней остановкой. Метрика - ROC-AUC на отложенной части.

In [20]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

TUNE_FOLD_NN = folds_nn[0]
TUNE_FOLD_GBDT = folds_gbdt[0]
TUNE_EPOCHS = 4
XGB_TRIALS = 20
NN_TRIALS = 12

In [21]:
def tune_xgboost(trial):
    params = dict(
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        max_depth=trial.suggest_int('max_depth', 4, 10),
        min_child_weight=trial.suggest_int('min_child_weight', 1, 30),
        subsample=trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.6, 1.0),
        reg_lambda=trial.suggest_float('reg_lambda', 0.1, 10.0, log=True),
        gamma=trial.suggest_float('gamma', 0.0, 5.0),
    )
    train_idx, val_idx = TUNE_FOLD_GBDT
    spw = float((y[train_idx] == 0).sum()) / max(float((y[train_idx] == 1).sum()), 1.0)
    model = XGBClassifier(
        n_estimators=3000, early_stopping_rounds=100, eval_metric='auc',
        scale_pos_weight=spw, tree_method='gpu_hist', random_state=SEED, n_jobs=-1, **params)
    model.fit(X_agg_np[train_idx], y[train_idx],
              eval_set=[(X_agg_np[val_idx], y[val_idx])], verbose=False)
    return roc_auc_score(y[val_idx], model.predict_proba(X_agg_np[val_idx])[:, 1])


def suggest_gru(trial):
    return dict(
        d_model=trial.suggest_categorical('d_model', [128, 192, 256, 320]),
        gru_layers=trial.suggest_int('gru_layers', 1, 3),
        dropout=trial.suggest_float('dropout', 0.0, 0.4),
        max_lr=trial.suggest_float('max_lr', 5e-4, 4e-3, log=True),
        weight_decay=trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True),
    )


def suggest_transformer(trial):
    return dict(
        d_model=trial.suggest_categorical('d_model', [128, 192, 256, 320]),
        tf_layers=trial.suggest_int('tf_layers', 2, 4),
        tf_ff=trial.suggest_categorical('tf_ff', [512, 768, 1024, 1536]),
        dropout=trial.suggest_float('dropout', 0.0, 0.3),
        max_lr=trial.suggest_float('max_lr', 5e-4, 4e-3, log=True),
        weight_decay=trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True),
    )


def suggest_cnn(trial):
    return dict(
        d_model=trial.suggest_categorical('d_model', [128, 192, 256, 320]),
        tcn_levels=trial.suggest_int('tcn_levels', 3, 6),
        dropout=trial.suggest_float('dropout', 0.0, 0.3),
        max_lr=trial.suggest_float('max_lr', 5e-4, 4e-3, log=True),
        weight_decay=trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True),
    )


def suggest_hier(trial):
    return dict(
        d_model=trial.suggest_categorical('d_model', [128, 192, 256]),
        hier_layers=trial.suggest_int('hier_layers', 2, 5),
        hier_hidden_mult=trial.suggest_int('hier_hidden_mult', 2, 3),
        dropout=trial.suggest_float('dropout', 0.0, 0.3),
        max_lr=trial.suggest_float('max_lr', 5e-4, 3e-3, log=True),
        weight_decay=trial.suggest_float('weight_decay', 1e-3, 5e-2, log=True),
        grad_clip=trial.suggest_categorical('grad_clip', [0.5, 1.0, 2.0]),
        label_smooth=trial.suggest_float('label_smooth', 0.0, 0.05),
    )


def tune_nn(trial, model_class, suggest):
    params = suggest(trial)
    model = model_class(params)
    model.epochs = TUNE_EPOCHS
    model.n_seeds = 1
    model.bs = BATCH
    model._K = 1
    model._k = 1
    model._ep_times = []
    model._ep_count = 0
    train_idx, val_idx = TUNE_FOLD_NN
    net, best_auc = model._train_one(train_idx, val_idx, 0)
    del net
    gc.collect()
    torch.cuda.empty_cache()
    return best_auc

In [22]:
def run_study(name, objective):
    sampler = optuna.samplers.TPESampler(seed=SEED)
    study = optuna.create_study(direction='maximize', sampler=sampler)
    n = XGB_TRIALS if name == 'xgboost' else NN_TRIALS
    print(f'=== тюнинг {name}: {n} трейлов ===')

    def progress(study, trial):
        value = trial.value if trial.value is not None else float('nan')
        print(f'  [{name}] трейл {trial.number + 1}/{n} | AUC={value:.4f} '
              f'| лучший={study.best_value:.4f}')

    study.optimize(objective, n_trials=n, gc_after_trial=True, callbacks=[progress])
    print(f'>> {name}: лучший AUC={study.best_value:.4f}')
    return study


nn_specs = [
    ('gru', GRUModel, suggest_gru),
    ('transformer', TransformerModel, suggest_transformer),
    ('cnn', TCNModel, suggest_cnn),
    ('transformer_hier', TransformerHierModel, suggest_hier),
]

studies = {}
studies['xgboost'] = run_study('xgboost', tune_xgboost)
for tune_name, tune_cls, tune_suggest in nn_specs:
    studies[tune_name] = run_study(
        tune_name, lambda t, c=tune_cls, s=tune_suggest: tune_nn(t, c, s))

=== тюнинг xgboost: 20 трейлов ===
  [xgboost] трейл 1/20 | AUC=0.7522 | лучший=0.7522
  [xgboost] трейл 2/20 | AUC=0.7517 | лучший=0.7522
  [xgboost] трейл 3/20 | AUC=0.7601 | лучший=0.7601
  [xgboost] трейл 4/20 | AUC=0.7596 | лучший=0.7601
  [xgboost] трейл 5/20 | AUC=0.7591 | лучший=0.7601
  [xgboost] трейл 6/20 | AUC=0.7576 | лучший=0.7601
  [xgboost] трейл 7/20 | AUC=0.7524 | лучший=0.7601
  [xgboost] трейл 8/20 | AUC=0.7526 | лучший=0.7601
  [xgboost] трейл 9/20 | AUC=0.7600 | лучший=0.7601
  [xgboost] трейл 10/20 | AUC=0.7594 | лучший=0.7601
  [xgboost] трейл 11/20 | AUC=0.7570 | лучший=0.7601
  [xgboost] трейл 12/20 | AUC=0.7597 | лучший=0.7601
  [xgboost] трейл 13/20 | AUC=0.7600 | лучший=0.7601
  [xgboost] трейл 14/20 | AUC=0.7593 | лучший=0.7601
  [xgboost] трейл 15/20 | AUC=0.7593 | лучший=0.7601
  [xgboost] трейл 16/20 | AUC=0.7595 | лучший=0.7601
  [xgboost] трейл 17/20 | AUC=0.7586 | лучший=0.7601
  [xgboost] трейл 18/20 | AUC=0.7592 | лучший=0.7601
  [xgboost] трейл 19

In [27]:
print('MODEL_PARAMS = {')
for model_name in ['xgboost', 'gru', 'transformer', 'cnn', 'transformer_hier']:
    print(f'    {model_name!r}: {studies[model_name].best_params},')
print('}')
print()
for model_name, study in studies.items():
    print(f'{model_name}: AUC={study.best_value:.4f}')

MODEL_PARAMS = {
    'xgboost': {'learning_rate': 0.015199348301309814, 'max_depth': 5, 'min_child_weight': 10, 'subsample': 0.8099025726528951, 'colsample_bytree': 0.7727780074568463, 'reg_lambda': 0.38234752246751863, 'gamma': 3.0592644736118975},
    'gru': {'d_model': 128, 'gru_layers': 1, 'dropout': 0.38056375658581343, 'max_lr': 0.0020898571378360426, 'weight_decay': 1.053746415699816e-06},
    'transformer': {'d_model': 128, 'tf_layers': 2, 'tf_ff': 1536, 'dropout': 0.07478766874466249, 'max_lr': 0.001173769209815432, 'weight_decay': 0.0001847793417351927},
    'cnn': {'d_model': 128, 'tcn_levels': 4, 'dropout': 0.12899315955145987, 'max_lr': 0.003478756568985834, 'weight_decay': 0.0001398005360250269},
    'transformer_hier': {'d_model': 192, 'hier_layers': 5, 'hier_hidden_mult': 3, 'dropout': 0.073378026190659, 'max_lr': 0.0028020568721807385, 'weight_decay': 0.0011277712372014764, 'grad_clip': 2.0, 'label_smooth': 0.04836599730858594},
}

xgboost: AUC=0.7601
gru: AUC=0.7858
t

## Подбор порогов псевдо-разметки

Широкий перебор на удешевлённом бустинге: варьируются порог позитивов, порог негативов и предельная доля негативов. Для каждой комбинации модель учится на части обучающих данных вместе с псевдо-размеченным тестом и проверяется на отложенных фолдах. качество усредняется по фолдам и сравнивается с тем же бустингом без псевдо-разметки.

In [28]:
from xgboost import XGBClassifier
from scipy.stats import rankdata

SWEEP_POS_Q = [0.70, 0.75, 0.80, 0.85, 0.90]
SWEEP_NEG_Q = [0.8, 0.9, 0.99]
SWEEP_MULT = [5, 10]
SWEEP_FOLDS = 2
SWEEP_TRAIN = 300_000
SWEEP_TREES = 300

rank_model = XGBClassifier(
    n_estimators=400, max_depth=7, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    tree_method='gpu_hist', random_state=SEED, n_jobs=-1)
rank_model.fit(X_agg_np, y)
test_proba = rank_model.predict_proba(X_test_agg_np)[:, 1]
test_rank = rankdata(test_proba) / len(test_proba)

sweep_rng = np.random.default_rng(SEED)
splits = folds_gbdt[:SWEEP_FOLDS]
subs = [sweep_rng.choice(tr, min(SWEEP_TRAIN, len(tr)), replace=False) for tr, _ in splits]

cheap = dict(n_estimators=SWEEP_TREES, max_depth=7, learning_rate=0.05,
             subsample=0.8, colsample_bytree=0.8,
             tree_method='gpu_hist', random_state=SEED, n_jobs=-1)


def cv_auc(extra_idx, extra_y):
    aucs = []
    for sub, (train_idx, val_idx) in zip(subs, splits):
        if extra_idx is None:
            X_tr = X_agg_np[sub]
            y_tr = y[sub]
        else:
            X_tr = np.vstack([X_agg_np[sub], X_test_agg_np[extra_idx]])
            y_tr = np.concatenate([y[sub], extra_y]).astype(np.int8)
        model = XGBClassifier(**cheap)
        model.fit(X_tr, y_tr)
        aucs.append(roc_auc_score(y[val_idx], model.predict_proba(X_agg_np[val_idx])[:, 1]))
    return float(np.mean(aucs)), float(np.std(aucs))


base_mean, base_std = cv_auc(None, None)
total = len(SWEEP_POS_Q) * len(SWEEP_NEG_Q) * len(SWEEP_MULT)
print(f'комбинаций {total} | фолдов {SWEEP_FOLDS} | база {base_mean:.4f} +/- {base_std:.4f}')

records = []
done = 0
sweep_start = time.time()
for pos_q in SWEEP_POS_Q:
    pos = np.where(test_rank >= pos_q)[0]
    for neg_q in SWEEP_NEG_Q:
        pool = np.where((test_rank <= neg_q) & (test_rank < pos_q))[0]
        for mult in SWEEP_MULT:
            n_neg = min(len(pool), max(1, len(pos)) * mult)
            neg = sweep_rng.choice(pool, n_neg, replace=False) if len(pool) else pool
            conf = np.concatenate([pos, neg]).astype(np.int64)
            pseudo = np.concatenate([np.ones(len(pos)), np.zeros(len(neg))]).astype(np.int8)
            mean, std = cv_auc(conf, pseudo)
            records.append({'pos_q': pos_q, 'neg_q': neg_q, 'mult': mult,
                            'n_pos': len(pos), 'n_neg': len(neg),
                            'auc': round(mean, 4), 'std': round(std, 4),
                            'delta': round(mean - base_mean, 4)})
            done += 1
            eta = (time.time() - sweep_start) / done * (total - done)
            print(f'  [{done}/{total}] pos_q={pos_q} +{len(pos)} -{len(neg)} '
                  f'| AUC={mean:.4f} ({mean - base_mean:+.4f}) | ~{eta / 60:.1f} мин')

table = pd.DataFrame(records).sort_values('auc', ascending=False).reset_index(drop=True)
print('\nрезультаты по порогу позитивов:')
print(table.to_string(index=False))
best = table.iloc[0]
print(f'\nлучшее: pos_q={best["pos_q"]} neg_q={best["neg_q"]} mult={int(best["mult"])} '
      f'-> AUC={best["auc"]:.4f} +/- {best["std"]:.4f} (прирост {best["delta"]:+.4f})')

комбинаций 30 | фолдов 2 | база 0.7457 +/- 0.0015
  [1/30] pos_q=0.7 +270001 -629999 | AUC=0.7567 (+0.0110) | ~13.0 мин
  [2/30] pos_q=0.7 +270001 -629999 | AUC=0.7567 (+0.0110) | ~12.5 мин
  [3/30] pos_q=0.7 +270001 -629999 | AUC=0.7569 (+0.0112) | ~12.1 мин
  [4/30] pos_q=0.7 +270001 -629999 | AUC=0.7569 (+0.0112) | ~11.6 мин
  [5/30] pos_q=0.7 +270001 -629999 | AUC=0.7567 (+0.0111) | ~11.2 мин
  [6/30] pos_q=0.7 +270001 -629999 | AUC=0.7568 (+0.0111) | ~10.7 мин
  [7/30] pos_q=0.75 +225001 -674999 | AUC=0.7575 (+0.0118) | ~10.3 мин
  [8/30] pos_q=0.75 +225001 -674999 | AUC=0.7573 (+0.0117) | ~9.9 мин
  [9/30] pos_q=0.75 +225001 -674999 | AUC=0.7574 (+0.0118) | ~9.4 мин
  [10/30] pos_q=0.75 +225001 -674999 | AUC=0.7574 (+0.0118) | ~9.0 мин
  [11/30] pos_q=0.75 +225001 -674999 | AUC=0.7574 (+0.0117) | ~8.5 мин
  [12/30] pos_q=0.75 +225001 -674999 | AUC=0.7573 (+0.0117) | ~8.1 мин
  [13/30] pos_q=0.8 +180001 -719999 | AUC=0.7581 (+0.0124) | ~7.6 мин
  [14/30] pos_q=0.8 +180001 -719999 